# ML-07 — Baseline Action Score and Top-20 Review

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
from dotenv import load_dotenv
import os
import pandas as pd
import numpy as np

load_dotenv()

HF_TOKEN = os.getenv("HF_TOKEN")
assert HF_TOKEN, "HF_TOKEN is not set."

print("HF token loaded successfully.")

HF token loaded successfully.


In [2]:
from huggingface_hub import hf_hub_download

march_file = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    filename="fact_content_daily_performance/month=2026-03/data_0.parquet",
    repo_type="dataset",
    token=HF_TOKEN
)

march_df = pd.read_parquet(march_file)

print("March 2026 rows:", len(march_df))
print("Columns:", list(march_df.columns))

D:\download_99\Anaconda\envs\Machine_Learning_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


March 2026 rows: 9841378
Columns: ['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events']


In [3]:
page_agg = (
    march_df
    .groupby(["client_hash_id", "content_hash_id"], as_index=False)
    .agg(
        march_gsc_impressions=("gsc_impressions", "sum"),
        march_gsc_clicks=("gsc_clicks", "sum"),
        _pos_count=("gsc_avg_position", "count"),
        _pos_sum=("gsc_avg_position", lambda x: x[x > 0].sum()),
        _pos_valid_n=("gsc_avg_position", lambda x: (x > 0).sum()),
    )
)

page_agg["march_gsc_avg_position"] = np.where(
    page_agg["_pos_valid_n"] > 0,
    page_agg["_pos_sum"] / page_agg["_pos_valid_n"],
    np.nan,
)
page_agg = page_agg.drop(columns=["_pos_count", "_pos_sum", "_pos_valid_n"])

page_agg["opportunity_proxy"] = (
    (page_agg["march_gsc_impressions"] > 0)
    & (page_agg["march_gsc_clicks"] == 0)
).astype(int)

print("Page-level records:", len(page_agg))
print("Unique clients:", page_agg["client_hash_id"].nunique())
print(f"Opportunity proxy base rate: {page_agg['opportunity_proxy'].mean():.1%}")
print(f"Pages with position data: {page_agg['march_gsc_avg_position'].notna().mean():.1%}")

Page-level records: 331437
Unique clients: 55
Opportunity proxy base rate: 32.6%
Pages with position data: 52.9%


## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

**Prioritize pages that have meaningful search visibility but rank poorly.** Use average search position as the main priority signal, with impression volume helping distinguish pages within the same position tier.

Specifically:
- Pages ranking deep in search results (position > 10) are prioritized, with deeper positions ranking higher.
- Within the same position tier, pages with more impressions rank higher because they have proven search demand.
- Pages ranking in the top 3 positions are deprioritized — they are already performing well.
- Pages with zero impressions or no position data are deprioritized — there is insufficient evidence of search relevance.

This rule does **not** use CTR, GA4 data, health scores, or any product-generated flags.

### Signal check #1: GSC Impression Volume

**Verdict from signal audit: CONFIRMED** (with caveat: partially mechanical overlap with the proxy, because impressions > 0 is part of the proxy definition).

In [4]:
imp_bins = [-0.01, 0, 50, 200, 1000, 5000, float("inf")]
imp_labels = ["0", "1-50", "51-200", "201-1000", "1001-5000", "5000+"]

tmp = page_agg.copy()
tmp["_bucket"] = pd.cut(tmp["march_gsc_impressions"], bins=imp_bins, labels=imp_labels, include_lowest=True)
imp_table = (
    tmp.groupby("_bucket", observed=False)
    .agg(n=("opportunity_proxy", "size"), opportunity_n=("opportunity_proxy", "sum"))
    .reset_index()
)
imp_table["opportunity_rate"] = imp_table["opportunity_n"] / imp_table["n"]
imp_table = imp_table.rename(columns={"_bucket": "bucket"})

print("Signal #1: GSC Impression Volume vs Opportunity Proxy")
print("-" * 60)
print(imp_table.to_string(index=False))
print()
print("VERDICT: CONFIRMED")
print("Caveat: partially mechanical — impressions > 0 is part of the proxy definition.")
print("This is NOT future leakage, but the signal is not fully independent of the proxy.")
print(f"Minimum bucket n = {imp_table['n'].min():,}. All buckets have sufficient sample size.")

Signal #1: GSC Impression Volume vs Opportunity Proxy
------------------------------------------------------------
   bucket      n  opportunity_n  opportunity_rate
        0 154699              0          0.000000
     1-50  61015          57877          0.948570
   51-200  31014          25213          0.812955
 201-1000  39674          20433          0.515022
1001-5000  31745           4139          0.130383
    5000+  13290            239          0.017983

VERDICT: CONFIRMED
Caveat: partially mechanical — impressions > 0 is part of the proxy definition.
This is NOT future leakage, but the signal is not fully independent of the proxy.
Minimum bucket n = 13,290. All buckets have sufficient sample size.


### Signal check #2: GSC Average Position

**Verdict from signal audit: CONFIRMED** — the strongest independent signal. Position is NOT a direct component of the proxy (which only uses impressions and clicks). Any relationship here represents genuinely independent signal.

In [5]:
pos_df = page_agg[page_agg["march_gsc_avg_position"].notna()].copy()

pos_bins = [0, 3, 10, 20, 50, float("inf")]
pos_labels = ["1-3 (top)", "4-10", "11-20", "21-50", "50+ (deep)"]

tmp2 = pos_df.copy()
tmp2["_bucket"] = pd.cut(tmp2["march_gsc_avg_position"], bins=pos_bins, labels=pos_labels, include_lowest=True)
pos_table = (
    tmp2.groupby("_bucket", observed=False)
    .agg(n=("opportunity_proxy", "size"), opportunity_n=("opportunity_proxy", "sum"))
    .reset_index()
)
pos_table["opportunity_rate"] = pos_table["opportunity_n"] / pos_table["n"]
pos_table = pos_table.rename(columns={"_bucket": "bucket"})

print("Signal #2: GSC Average Position vs Opportunity Proxy")
print(f"(Tested on {len(pos_df):,} pages with valid position data)")
print("-" * 60)
print(pos_table.to_string(index=False))
print()
print("VERDICT: CONFIRMED")
print("Position is independent of the proxy definition — it uses only impressions and clicks.")
print("This is the strongest independent signal for the baseline rule.")
print(f"Minimum bucket n = {pos_table['n'].min():,}.")

Signal #2: GSC Average Position vs Opportunity Proxy
(Tested on 175,304 pages with valid position data)
------------------------------------------------------------
    bucket     n  opportunity_n  opportunity_rate
 1-3 (top) 13136           5814          0.442600
      4-10 81619          44661          0.547189
     11-20 32548          19253          0.591526
     21-50 34783          24184          0.695282
50+ (deep) 13218          12612          0.954153

VERDICT: CONFIRMED
Position is independent of the proxy definition — it uses only impressions and clicks.
This is the strongest independent signal for the baseline rule.
Minimum bucket n = 13,136.


### Reason codes

Every row gets exactly ONE reason code:

| Reason code | Meaning |
|---|---|
| `poor_position_high_visibility` | Position > 50 AND impressions > 1000 |
| `poor_position_moderate_visibility` | Position > 50 AND impressions 201-1000 |
| `poor_position_low_visibility` | Position > 50 AND impressions 1-200 |
| `moderate_position_high_visibility` | Position 21-50 AND impressions > 1000 |
| `moderate_position_moderate_visibility` | Position 21-50 AND impressions 201-1000 |
| `moderate_position_low_visibility` | Position 21-50 AND impressions 1-200 |
| `mid_position_moderate_visibility` | Position 11-20 AND impressions > 200 |
| `mid_position_low_visibility` | Position 11-20 AND impressions 1-200 |
| `good_position` | Position 1-10 |
| `insufficient_visibility` | Impressions = 0 |
| `position_unavailable` | Position data missing |

### Action labels

| Action | Meaning |
|---|---|
| `REVIEW` | High priority — strong signal of missed search opportunity |
| `MONITOR` | Medium priority — some signal but less certain |
| `DEPRIORITISE` | Low priority — insufficient signal or already performing well |

### Score formula

```
position_score = 0   if pos <= 3
                 30  if pos 4-10
                 60  if pos 11-20
                 80  if pos 21-50
                 100 if pos > 50
                 0   if pos missing

impression_score = 0  if imp = 0
                   10  if imp 1-200
                   20  if imp 201-1000
                   30  if imp > 1000

score = position_score + impression_score
```

Position drives 0-100 of the score (77% of max). Impressions contribute 0-30 (23% of max). This ensures position is the dominant ranking signal, with impressions acting as a secondary tiebreaker within position tiers.

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [6]:
def position_score(pos):
    """Score based on average search position. Higher = worse rank = more opportunity."""
    if pd.isna(pos):
        return 0
    if pos <= 3:
        return 0
    elif pos <= 10:
        return 30
    elif pos <= 20:
        return 60
    elif pos <= 50:
        return 80
    else:
        return 100

def impression_score(imp):
    """Score based on impression volume. Higher = more search demand."""
    if imp == 0:
        return 0
    elif imp <= 200:
        return 10
    elif imp <= 1000:
        return 20
    else:
        return 30

def reason_code(row):
    """Assign exactly one reason code per row."""
    pos = row["march_gsc_avg_position"]
    imp = row["march_gsc_impressions"]

    if pd.isna(pos):
        return "position_unavailable"
    if imp == 0:
        return "insufficient_visibility"
    if pos <= 10:
        return "good_position"
    if pos <= 20:
        if imp > 200:
            return "mid_position_moderate_visibility"
        return "mid_position_low_visibility"
    if pos <= 50:
        if imp > 1000:
            return "moderate_position_high_visibility"
        if imp > 200:
            return "moderate_position_moderate_visibility"
        return "moderate_position_low_visibility"
    # pos > 50
    if imp > 1000:
        return "poor_position_high_visibility"
    if imp > 200:
        return "poor_position_moderate_visibility"
    return "poor_position_low_visibility"

def action_label(score):
    """Map score to action."""
    if score >= 60:
        return "REVIEW"
    elif score >= 30:
        return "MONITOR"
    else:
        return "DEPRIORITISE"

print("Scoring functions defined.")

Scoring functions defined.


In [7]:
page_agg["position_score"] = page_agg["march_gsc_avg_position"].apply(position_score)
page_agg["impression_score"] = page_agg["march_gsc_impressions"].apply(impression_score)
page_agg["score"] = page_agg["position_score"] + page_agg["impression_score"]
page_agg["reason_code"] = page_agg.apply(reason_code, axis=1)
page_agg["action"] = page_agg["score"].apply(action_label)

# Rank: highest score first, then stable tie-breaker (client_hash_id, content_hash_id)
page_agg = page_agg.sort_values(
    ["score", "client_hash_id", "content_hash_id"],
    ascending=[False, True, True]
).reset_index(drop=True)
page_agg["rank"] = range(1, len(page_agg) + 1)

print("Scoring complete.")
print(f"Total pages: {len(page_agg):,}")
print(f"Score distribution:")
print(page_agg["score"].describe())
print(f"\nAction distribution:")
print(page_agg["action"].value_counts())
print(f"\nReason code distribution:")
print(page_agg["reason_code"].value_counts())

Scoring complete.
Total pages: 331,437
Score distribution:
count    331437.000000
mean         34.910737
std          38.403528
min           0.000000
25%           0.000000
50%          30.000000
75%          60.000000
max         130.000000
Name: score, dtype: float64

Action distribution:
action
DEPRIORITISE    163484
REVIEW          104338
MONITOR          63615
Name: count, dtype: int64

Reason code distribution:
reason_code
position_unavailable                     156133
good_position                             94755
moderate_position_low_visibility          18763
mid_position_moderate_visibility          17865
mid_position_low_visibility               14683
poor_position_low_visibility              11272
moderate_position_moderate_visibility      8382
moderate_position_high_visibility          7638
poor_position_moderate_visibility          1709
poor_position_high_visibility               237
Name: count, dtype: int64


In [8]:
output_cols = ["client_hash_id", "content_hash_id", "score", "reason_code", "action", "rank"]
output_df = page_agg[output_cols].copy()

_proj_root = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
output_path = os.path.join(_proj_root, "work", "outputs", "baseline_action_score.csv")
os.makedirs(os.path.dirname(output_path), exist_ok=True)
output_df.to_csv(output_path, index=False)

print(f"CSV written: {output_path}")
print(f"Rows: {len(output_df):,}")
print(f"Unique ranks: {output_df['rank'].nunique():,}")
print(f"Any duplicate (client, content): {output_df.duplicated(subset=['client_hash_id', 'content_hash_id']).sum()}")

CSV written: D:\download_99\Dreams\Interns\flyrank.ai intern\Flyrank_ML_Assign\work\outputs\baseline_action_score.csv
Rows: 331,437
Unique ranks: 331,437
Any duplicate (client, content): 0


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [9]:
top20 = page_agg.head(20).copy()

def confidence_note(row):
    """Provide a confidence note based on available data."""
    pos = row["march_gsc_avg_position"]
    imp = row["march_gsc_impressions"]
    clicks = row["march_gsc_clicks"]
    pos_valid = row.get("_pos_valid_n", 0) if "_pos_valid_n" in row.index else 0

    notes = []
    if imp < 50:
        notes.append(f"Low impression volume ({imp}) — limited search demand evidence")
    if imp > 5000:
        notes.append(f"High impression volume ({imp:,}) — strong search demand")
    if pd.isna(pos):
        notes.append("No position data — ranking context unknown")
    if clicks > 0:
        notes.append(f"Has {clicks} clicks — page is generating some traffic")
    if not notes:
        notes.append("Moderate data — reasonable signal")
    return "; ".join(notes)

def what_would_make_it_wrong(row):
    """Identify what would invalidate this recommendation."""
    pos = row["march_gsc_avg_position"]
    imp = row["march_gsc_impressions"]
    clicks = row["march_gsc_clicks"]

    issues = []
    if imp < 50:
        issues.append("Very low impressions suggest the page may not have real search demand")
    if pd.isna(pos):
        issues.append("Missing position data means we cannot confirm the page ranks at all")
    if clicks > 0:
        issues.append("Page already gets clicks — may not need a refresh")
    if imp > 10000 and not pd.isna(pos) and pos < 10:
        issues.append("High impressions with good position — page may already be performing well")
    if not issues:
        issues.append("Position and impression data are moderate — a human SEO review should confirm priority")
    return "; ".join(issues)

print("=" * 80)
print("TOP-20 REVIEW")
print("=" * 80)
print()

for idx, row in top20.iterrows():
    rank = row["rank"]
    print(f"--- Rank #{rank} ---")
    print(f"  client_hash_id:   {row['client_hash_id']}")
    print(f"  content_hash_id:  {row['content_hash_id']}")
    print(f"  score:            {row['score']}")
    print(f"  action:           {row['action']}")
    print(f"  reason_code:      {row['reason_code']}")
    print(f"  impressions:      {row['march_gsc_impressions']:,}")
    print(f"  clicks:           {row['march_gsc_clicks']:,}")
    print(f"  avg_position:     {row['march_gsc_avg_position']:.1f}" if not pd.isna(row['march_gsc_avg_position']) else "  avg_position:     N/A")
    print(f"  confidence:       {confidence_note(row)}")
    print(f"  what_wrong:       {what_would_make_it_wrong(row)}")
    print()

TOP-20 REVIEW

--- Rank #1 ---
  client_hash_id:   client_0797ff3a1fc9a6a5
  content_hash_id:  content_be06033d30b49299
  score:            130
  action:           REVIEW
  reason_code:      poor_position_high_visibility
  impressions:      2,092
  clicks:           1
  avg_position:     51.7
  confidence:       Has 1 clicks — page is generating some traffic
  what_wrong:       Page already gets clicks — may not need a refresh

--- Rank #2 ---
  client_hash_id:   client_08a6a72ff48e62c0
  content_hash_id:  content_1c13a0a5eb12cbb5
  score:            130
  action:           REVIEW
  reason_code:      poor_position_high_visibility
  impressions:      1,504
  clicks:           0
  avg_position:     52.4
  confidence:       Moderate data — reasonable signal
  what_wrong:       Position and impression data are moderate — a human SEO review should confirm priority

--- Rank #3 ---
  client_hash_id:   client_08a6a72ff48e62c0
  content_hash_id:  content_1c2f1ad5b615ea6a
  score:            13

### Top-20 observations

*Look for: low impression counts, missing position, extreme position values, client concentration, pages with legitimate clicks, threshold artifacts, suspiciously weak recommendations.*

In [10]:
print("=" * 80)
print("TOP-20 PATTERN ANALYSIS")
print("=" * 80)
print()

# Client concentration
client_counts = top20["client_hash_id"].value_counts()
print("Client concentration in top 20:")
for c, n in client_counts.items():
    print(f"  {c}: {n} pages")
print()

# Impression range
print(f"Impression range: {top20['march_gsc_impressions'].min()} to {top20['march_gsc_impressions'].max():,}")
print(f"Median impressions: {top20['march_gsc_impressions'].median():,.0f}")
print()

# Position range
pos_valid = top20[top20["march_gsc_avg_position"].notna()]
if len(pos_valid) > 0:
    print(f"Position range (where available): {pos_valid['march_gsc_avg_position'].min():.1f} to {pos_valid['march_gsc_avg_position'].max():.1f}")
    print(f"Median position: {pos_valid['march_gsc_avg_position'].median():.1f}")
else:
    print("No valid position data in top 20")
print()

# Pages with clicks
pages_with_clicks = top20[top20["march_gsc_clicks"] > 0]
print(f"Pages with clicks in top 20: {len(pages_with_clicks)}")
if len(pages_with_clicks) > 0:
    print("  These pages already receive some traffic — refresh may be less urgent.")
print()

# Reason code breakdown
print("Reason code breakdown in top 20:")
for rc, n in top20["reason_code"].value_counts().items():
    print(f"  {rc}: {n}")
print()

# Weak signals
low_imp = top20[top20["march_gsc_impressions"] < 50]
print(f"Pages with < 50 impressions in top 20: {len(low_imp)}")
if len(low_imp) > 0:
    print("  These may be weak picks — low impression counts suggest limited search demand.")
no_pos = top20[top20["march_gsc_avg_position"].isna()]
print(f"Pages with missing position in top 20: {len(no_pos)}")

TOP-20 PATTERN ANALYSIS

Client concentration in top 20:
  client_08a6a72ff48e62c0: 17 pages
  client_0797ff3a1fc9a6a5: 1 pages
  client_157ffe4d4a595515: 1 pages
  client_1a730cb2640a1abf: 1 pages

Impression range: 1030 to 3,063
Median impressions: 1,209

Position range (where available): 51.7 to 78.8
Median position: 63.8

Pages with clicks in top 20: 3
  These pages already receive some traffic — refresh may be less urgent.

Reason code breakdown in top 20:
  poor_position_high_visibility: 20

Pages with < 50 impressions in top 20: 0
Pages with missing position in top 20: 0


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [11]:
print("=" * 80)
print("WEAK PICKS ANALYSIS")
print("=" * 80)
print()

# Find pages that look like weak picks
# 1. Pages with low impressions but high scores (driven by position alone)
weak_low_imp = page_agg[
    (page_agg["score"] >= 60) &
    (page_agg["march_gsc_impressions"] < 50) &
    (page_agg["march_gsc_avg_position"].notna()) &
    (page_agg["march_gsc_avg_position"] > 50)
].head(5)

print("Weak pick type 1: Deep position but very low impressions")
print("These pages rank poorly (position > 50) but have fewer than 50 impressions.")
print("A human might question whether the page has real search demand.")
print()
if len(weak_low_imp) > 0:
    for _, row in weak_low_imp.iterrows():
        print(f"  rank {row['rank']}: pos={row['march_gsc_avg_position']:.1f}, imp={row['march_gsc_impressions']}, score={row['score']}")
        print(f"    Why rule selected it: Deep position scores 100 position points, plus impression points")
        print(f"    Why human might disagree: Very few impressions suggest limited search demand")
        print(f"    What would resolve: Check if the page has any meaningful search queries or topical relevance")
        print()
else:
    print("  (No pages matching this pattern found)")
    print()

# 2. Pages with clicks already
weak_with_clicks = page_agg[
    (page_agg["score"] >= 60) &
    (page_agg["march_gsc_clicks"] > 0) &
    (page_agg["march_gsc_avg_position"].notna()) &
    (page_agg["march_gsc_avg_position"] > 50)
].head(5)

print("Weak pick type 2: Deep position but already receiving clicks")
print("These pages rank poorly but are generating some traffic.")
print("A human might argue these pages are already working and need less attention.")
print()
if len(weak_with_clicks) > 0:
    for _, row in weak_with_clicks.iterrows():
        print(f"  rank {row['rank']}: pos={row['march_gsc_avg_position']:.1f}, imp={row['march_gsc_impressions']}, clicks={row['march_gsc_clicks']}, score={row['score']}")
        print(f"    Why rule selected it: Deep position scores high, page still ranks poorly")
        print(f"    Why human might disagree: Page gets clicks — it may already be performing acceptably")
        print(f"    What would resolve: Check click quality and whether the page could rank better with content changes")
        print()
else:
    print("  (No pages matching this pattern found)")
    print()

WEAK PICKS ANALYSIS

Weak pick type 1: Deep position but very low impressions
These pages rank poorly (position > 50) but have fewer than 50 impressions.
A human might question whether the page has real search demand.

  rank 1948: pos=51.8, imp=14, score=110
    Why rule selected it: Deep position scores 100 position points, plus impression points
    Why human might disagree: Very few impressions suggest limited search demand
    What would resolve: Check if the page has any meaningful search queries or topical relevance

  rank 1949: pos=69.1, imp=46, score=110
    Why rule selected it: Deep position scores 100 position points, plus impression points
    Why human might disagree: Very few impressions suggest limited search demand
    What would resolve: Check if the page has any meaningful search queries or topical relevance

  rank 1950: pos=51.0, imp=11, score=110
    Why rule selected it: Deep position scores 100 position points, plus impression points
    Why human might disagre

In [12]:
print("=" * 80)
print("LEAKAGE AUDIT")
print("=" * 80)
print()

# Check 1: No future-window inputs
print("1. Future-window inputs:")
print("   PASS — All features use March 2026 data only.")
print("   No April or May data is loaded or used in scoring.")
print()

# Check 2: No label-derived inputs
print("2. Label-derived inputs:")
print("   PASS — The opportunity proxy is computed as an OUTCOME for evaluation only.")
print("   It is NOT used as a feature or input to the scoring rule.")
print("   The score depends only on position and impressions.")
print()

# Check 3: No product flags
print("3. Product flags:")
print("   PASS — No health_score, existing action labels, or product-generated")
print("   flags are used as features in the scoring rule.")
print("   The FlyRank flag relationship is acknowledged for signal explanation")
print("   but the existing product flag itself is NOT a feature.")
print()

# Check 4: No existing priority/action outputs
print("4. Existing priority/action outputs:")
print("   PASS — No pre-existing priority scores or action labels are used as inputs.")
print("   The action labels (REVIEW/MONITOR/DEPRIORITISE) are generated by this rule.")
print()

# Check 5: No proxy used as score
print("5. Proxy directly used as score:")
print("   PASS — The opportunity proxy is NOT used as the score.")
print("   The score is position_score + impression_score (0-130 scale).")
print("   The proxy is only used for signal evaluation, not scoring.")
print()

# Check 6: No learned weights
print("6. Learned weights:")
print("   PASS — All thresholds and weights are hand-written and fixed.")
print("   No logistic regression, random forest, XGBoost, or neural network.")
print("   No optimization or fitting on data.")
print()

# Check 7: Future data hidden in aggregation
print("7. Future data hidden in aggregation:")
print("   PASS — The page-level aggregation sums March daily rows only.")
"""No April or May data enters the aggregation."""
print("   No April or May data enters the aggregation.")
print()

# Mechanical overlap caveat
print("IMPORTANT CAVEAT:")
print("Impression volume has mechanical overlap with the opportunity proxy")
print("because impressions > 0 is part of the proxy definition.")
print("This is a limitation of using impression volume as a signal,")
print("NOT future leakage. The relationship is partially built into the")
print("data structure, but the impression score adds practical value")
print("for distinguishing pages within the same position tier.")

LEAKAGE AUDIT

1. Future-window inputs:
   PASS — All features use March 2026 data only.
   No April or May data is loaded or used in scoring.

2. Label-derived inputs:
   PASS — The opportunity proxy is computed as an OUTCOME for evaluation only.
   It is NOT used as a feature or input to the scoring rule.
   The score depends only on position and impressions.

3. Product flags:
   PASS — No health_score, existing action labels, or product-generated
   flags are used as features in the scoring rule.
   The FlyRank flag relationship is acknowledged for signal explanation
   but the existing product flag itself is NOT a feature.

4. Existing priority/action outputs:
   PASS — No pre-existing priority scores or action labels are used as inputs.
   The action labels (REVIEW/MONITOR/DEPRIORITISE) are generated by this rule.

5. Proxy directly used as score:
   PASS — The opportunity proxy is NOT used as the score.
   The score is position_score + impression_score (0-130 scale).
   The prox

## Self-check

Before you submit, confirm each line honestly:

- [✓] Every section above is filled — markdown thinking AND the code that backs it
- [✓] The notebook runs top to bottom with no errors (Runtime → Run all)
- [✓] No client names, URLs, or private queries anywhere
- [✓] My claims use careful words: observed, measured, directional, decision-support
- [✓] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.